In [16]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


In [17]:
df = pd.read_csv('../data/features/features_complete.csv')
day_map = {
    'monday': 0,
    'tuesday': 1,
    'wednesday': 2,
    'thursday': 3,
    'friday': 4,
    'saturday': 5,
    'sunday': 6
}
df['published_day_of_week_num'] = df['published_day_of_week'].str.lower().map(day_map)

features = [
    # Current features (keep these)
    'title_length', 
    'uppercase_words', 
    'sentiment_polarity', 
    'sentiment_subjectivity',
    'category_id',  
    'published_day_of_week_num',
    
    # Add these temporal features
    'hour_of_trending',       # Time of day
    'days_until_trending',    # How long to trend
    
    # Add these title features
    'num_emojis',
    'has_emoji',
    'contains_numbers_or_emojis',
    
    # Add these boolean flags
    'comments_disabled',
    'ratings_disabled',
    'is_english',
]
X = df[features]
y = np.log1p(df['views'])
print(f"Shape: X={X.shape}, y={y.shape}")

Shape: X=(26678, 14), y=(26678,)


In [18]:
xgb_model = xgb.XGBRegressor(
    random_state=42,
    verbosity=0  # Suppress training output
)
X_train, X_test, y_train, y_test  = train_test_split(X,y,test_size=0.2, random_state=42)


In [19]:

xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [20]:
y_pred_xgb = xgb_model.predict(X_test)

In [21]:
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"\nXGBoost Model Performance (LOG SCALE):")
print(f"RMSE: {rmse_xgb:.4f}")
print(f"MAE:  {mae_xgb:.4f}")
print(f"R²:   {r2_xgb:.4f}")



XGBoost Model Performance (LOG SCALE):
RMSE: 1.3463
MAE:  1.0403
R²:   0.2244


In [22]:
print("\n" + "="*60)
print("XGBOOST FEATURE IMPORTANCE")
print("="*60)

# Get feature importance scores
importance_scores = xgb_model.feature_importances_

# Create list of (feature, importance) tuples and sort
feature_importance = list(zip(features, importance_scores))
feature_importance.sort(key=lambda x: x[1], reverse=True)

print("\nTop Features (sorted by importance):")
for i, (feature, importance) in enumerate(feature_importance, 1):
    bar = '█' * int(importance * 100)
    print(f"{i:2d}. {feature:<30} {importance:>6.4f} {bar}")


XGBOOST FEATURE IMPORTANCE

Top Features (sorted by importance):
 1. num_emojis                     0.1301 █████████████
 2. comments_disabled              0.1265 ████████████
 3. days_until_trending            0.1248 ████████████
 4. uppercase_words                0.1145 ███████████
 5. category_id                    0.1002 ██████████
 6. sentiment_subjectivity         0.0831 ████████
 7. contains_numbers_or_emojis     0.0712 ███████
 8. hour_of_trending               0.0642 ██████
 9. ratings_disabled               0.0565 █████
10. published_day_of_week_num      0.0444 ████
11. title_length                   0.0429 ████
12. sentiment_polarity             0.0414 ████
13. has_emoji                      0.0000 
14. is_english                     0.0000 


In [23]:
from sklearn.model_selection import RandomizedSearchCV

# Define parameter distributions for RandomizedSearchCV
param_distributions = {
    'n_estimators': [100, 200, 300, 500],           # Number of trees
    'max_depth': [3, 5, 7, 9, 12],                  # Tree depth
    'learning_rate': [0.01, 0.05, 0.1, 0.2],        # Step size
    'subsample': [0.6, 0.8, 1.0],                   # Sample ratio per tree
    'colsample_bytree': [0.6, 0.8, 1.0],            # Feature ratio per tree
    'min_child_weight': [1, 3, 5, 7],               # Minimum samples in leaf
    'gamma': [0, 0.1, 0.2, 0.5],                    # Regularization parameter
}

print("Parameter grid defined!")
print(f"Total combinations: {4*5*4*3*3*4*4} ")

Parameter grid defined!
Total combinations: 11520 


In [24]:
xgb_base = xgb.XGBRegressor(
    randomstate =42,
    verbosity=0,
    n_jobs= -1
)

In [26]:
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=50,  
    cv=3,     
    scoring='neg_mean_squared_error', 
    random_state=42,
    verbose=2, 
    n_jobs=-1   
)
print("\nRandomizedSearchCV configured!")
print(f"Will test {random_search.n_iter} random parameter combinations")
print(f"Using {random_search.cv}-fold cross-validation")


RandomizedSearchCV configured!
Will test 50 random parameter combinations
Using 3-fold cross-validation


In [27]:
random_search.fit(X_train, y_train)

print("\n✅ Random Search fitting complete!")

best_params = random_search.best_params_
print("\n" + "="*60)
print("BEST HYPERPARAMETERS FOUND")
print("="*60)
for param, value in best_params.items():
    print(f"  {param:<20}: {value}")

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.1, max_depth=7, min_child_weight=3, n_estimators=100, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.1, max_depth=7, min_child_weight=3, n_estimators=100, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.1, max_depth=7, min_child_weight=3, n_estimators=100, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.1, max_depth=5, min_child_weight=3, n_estimators=500, subsample=0.8; total time=   0.4s
[CV] END colsample_bytree=0.6, gamma=0, learning_rate=0.2, max_depth=7, min_child_weight=7, n_estimators=300, subsample=1.0; total time=   0.3s
[CV] END colsample_bytree=0.6, gamma=0, learning_rate=0.2, max_depth=7, min_child_weight=7, n_estimators=300, subsample=1.0; total time=   0.2s
[CV] END colsample_bytree=0.6, gamma=0, learning_rate=0.2, max_dep

In [30]:
best_xgb = random_search.best_estimator_
y_pred_tuned = best_xgb.predict(X_test)

In [31]:
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

print("\n" + "="*60)
print("TUNED XGBOOST PERFORMANCE")
print("="*60)
print(f"RMSE: {rmse_tuned:.4f}")
print(f"MAE:  {mae_tuned:.4f}")
print(f"R²:   {r2_tuned:.4f}")

print(f"\nImprovement over default XGBoost:")
print(f"  RMSE: {((rmse_xgb - rmse_tuned) / rmse_xgb * 100):.2f}%")
print(f"  R² increase: {(r2_tuned - r2_xgb):.4f}")


TUNED XGBOOST PERFORMANCE
RMSE: 1.3285
MAE:  1.0314
R²:   0.2448

Improvement over default XGBoost:
  RMSE: 1.33%
  R² increase: 0.0204


Exception ignored in: <function ResourceTracker.__del__ at 0x106319d00>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102a1dd00>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x112755d00>
Traceback (most recent call last

In [33]:
print("\n" + "="*60)
print("Tuned XGBOOST  FEATURE IMPORTANCE")
print("="*60)

# Get feature importance scores
importance_scores = best_xgb.feature_importances_

# Create list of (feature, importance) tuples and sort
feature_importance = list(zip(features, importance_scores))
feature_importance.sort(key=lambda x: x[1], reverse=True)

print("\nTop Features (sorted by importance):")
for i, (feature, importance) in enumerate(feature_importance, 1):
    bar = '█' * int(importance * 100)
    print(f"{i:2d}. {feature:<30} {importance:>6.4f} {bar}")


Tuned XGBOOST  FEATURE IMPORTANCE

Top Features (sorted by importance):
 1. num_emojis                     0.1601 ████████████████
 2. days_until_trending            0.1294 ████████████
 3. comments_disabled              0.1137 ███████████
 4. category_id                    0.0852 ████████
 5. uppercase_words                0.0822 ████████
 6. sentiment_subjectivity         0.0782 ███████
 7. ratings_disabled               0.0701 ███████
 8. hour_of_trending               0.0683 ██████
 9. contains_numbers_or_emojis     0.0681 ██████
10. published_day_of_week_num      0.0503 █████
11. title_length                   0.0472 ████
12. sentiment_polarity             0.0472 ████
13. has_emoji                      0.0000 
14. is_english                     0.0000 
